# 01 - Preprocessing Bebas-Kebocoran (Paper 4 XAI)

Menyiapkan **CIC-IDS2017** (referensi/latih) dan **CSE-CIC-IDS2018** (target transfer) untuk eksperimen lintas-dataset biner, TANPA Semantic Feature Mapping. Arah utama **2017 -> 2018** (replikasi langsung paper referensi JTIIK).

**Prinsip:**
- Label biner: Benign/Normal = 0, Attack = 1.
- Penyelarasan fitur = IRISAN nama kolom setelah normalisasi (bukan SFM, bukan isi-nol).
- Semua statistik fitting (median imputasi, scaler, seleksi fitur, SMOTE) HANYA pada train. Dilakukan di notebook 02, bukan di sini.
- Output: artefak `cic2017.pkl` / `cic2018.pkl` + `schema_summary.json` untuk Tabel 1 paper.

## Dijalankan di AWS SageMaker
Notebook ini dieksekusi di SageMaker (bukan lokal). Alur: push ke GitHub -> `git pull` di SageMaker -> run -> upload `artifacts/` ke S3 -> download ke lokal untuk mengisi paper.

> **ISI SEL KONFIG S3.** Dataset mentah diambil dari S3; hasil di-upload balik ke S3.


In [ ]:
# === Instal paket (SageMaker) ===
# Jalankan sekali per kernel. Aman untuk dijalankan ulang.
import sys
!{sys.executable} -m pip install -q numpy pandas scipy scikit-learn xgboost imbalanced-learn shap
print('deps OK')


In [ ]:
# === KONFIG S3 + PATH (SageMaker) ===
import glob, os, json, pickle, gc, warnings, subprocess
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# ---- Bucket & prefix S3 (ISI sesuai bucket mitra AWS kamu) ----
S3_BUCKET        = 'ssh-detection-features-232032302717'  # bucket riset (ap-southeast-1)
S3_RAW_CIC2017   = 's3://%s/datasets/CICD2017/' % S3_BUCKET   # folder CSV CICFlowMeter 2017
S3_RAW_CIC2018   = 's3://%s/datasets/CICD2018/' % S3_BUCKET  # folder CSV CICFlowMeter 2018
S3_ARTIFACTS_OUT = 's3://%s/paper4-xai/artifacts/' % S3_BUCKET  # tujuan upload hasil

# ---- Folder lokal di instance SageMaker ----
DATA_DIR = './data'
CIC2017_DIR = os.path.join(DATA_DIR, 'cic2017')
CIC2018_DIR = os.path.join(DATA_DIR, 'cic2018')
OUT_DIR = './artifacts'
for d in (CIC2017_DIR, CIC2018_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

# AUTO_SYNC: cek folder lokal dulu; download dari S3 HANYA jika kosong.
# Set paksa-download dgn FORCE_SYNC=True (mis. ingin ambil ulang).
FORCE_SYNC = False

# Pola glob file CSV (setelah sync). Rekursif agar menangkap subfolder.
CIC2017_GLOB = os.path.join(CIC2017_DIR, '**', '*.csv')
CIC2018_GLOB = os.path.join(CIC2018_DIR, '**', '*.csv')

MAX_ROWS_PER_FILE = 100_000   # stratified cap per file (kendali memori)
SEED = 42
np.random.seed(SEED)
print('Config OK | bucket =', S3_BUCKET)


In [ ]:
# === Sync dataset dari S3 HANYA bila folder lokal kosong (hemat waktu/bandwidth) ===
def has_csv(d):
    return len(glob.glob(os.path.join(d, '**', '*.csv'), recursive=True)) > 0

def s3_sync(src, dst):
    print('sync', src, '->', dst)
    subprocess.run(['aws', 's3', 'sync', src, dst, '--only-show-errors'], check=True)

def ensure_local(s3_src, local_dir, name):
    if (not FORCE_SYNC) and has_csv(local_dir):
        n = len(glob.glob(os.path.join(local_dir, '**', '*.csv'), recursive=True))
        print(f'[{name}] {n} CSV sudah ada di {local_dir} -> lewati download')
    else:
        s3_sync(s3_src, local_dir)
        print(f'[{name}] selesai sync dari S3')

ensure_local(S3_RAW_CIC2017, CIC2017_DIR, 'CIC2017')
ensure_local(S3_RAW_CIC2018, CIC2018_DIR, 'CIC2018')
print('dataset siap di lokal')


In [ ]:
# === Util normalisasi kolom & label ===
META_SUBSTR = ['flow id','flowid','source ip','src ip','destination ip','dst ip',
               'source port','src port','destination port','dst port','protocol',
               'timestamp','unnamed']

def norm_colname(c):
    c = str(c).replace('\ufeff','').strip()
    c = ' '.join(c.split())           # rapatkan whitespace ganda
    return c.lower()

def find_label_col(cols):
    cand = [c for c in cols if 'label' in c]
    return cand[0] if cand else cols[-1]

def to_binary(label):
    s = str(label).strip().lower()
    return 0 if ('benign' in s or s == 'normal' or s == '0') else 1

def is_meta(c):
    return any(m in c for m in META_SUBSTR)

def load_glob(pattern, name):
    files = sorted(glob.glob(pattern, recursive=True))
    assert files, f'Tidak ada file untuk {name}: {pattern}'
    print(f'[{name}] {len(files)} file ditemukan')
    frames = []
    for fp in files:
        df = pd.read_csv(fp, low_memory=False)
        df.columns = [norm_colname(c) for c in df.columns]
        lab = find_label_col(list(df.columns))
        df = df.rename(columns={lab: 'label'})
        # stratified cap per file
        if len(df) > MAX_ROWS_PER_FILE:
            frac = MAX_ROWS_PER_FILE / len(df)
            df = df.groupby('label', group_keys=False).apply(
                lambda g: g.sample(max(1, int(len(g)*frac)), random_state=SEED))
        frames.append(df)
        print(f'   {os.path.basename(fp)}: {len(df):,} baris (setelah cap)')
    out = pd.concat(frames, ignore_index=True)
    del frames; gc.collect()
    return out

print('Util OK')


In [ ]:
# === Muat kedua dataset ===
cic17 = load_glob(CIC2017_GLOB, 'CIC2017')
cic18 = load_glob(CIC2018_GLOB, 'CIC2018')
print('raw shapes:', cic17.shape, cic18.shape)


In [ ]:
# === Bersihkan + label biner + ruang fitur numerik (tanpa metadata) ===
def clean(df):
    y = df['label'].apply(to_binary).astype(int).values
    X = df.drop(columns=['label'])
    keep = [c for c in X.columns if not is_meta(c)]   # buang metadata
    X = X[keep]
    X = X.apply(pd.to_numeric, errors='coerce')       # numerik saja
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.dropna(axis=1, how='all')                   # buang kolom semua-NaN
    return X, y

Xref, yref = clean(cic17)   # 2017 = referensi/latih
Xtgt, ytgt = clean(cic18)   # 2018 = target transfer
print('CIC2017 (ref):', Xref.shape, 'attack%=', round(100*yref.mean(),2))
print('CIC2018 (tgt):', Xtgt.shape, 'attack%=', round(100*ytgt.mean(),2))


In [ ]:
# === Penyelarasan fitur: IRISAN nama kolom (tanpa SFM, tanpa isi-nol) ===
common = sorted(set(Xref.columns) & set(Xtgt.columns))
print(f'Fitur CIC2017={Xref.shape[1]}, CIC2018={Xtgt.shape[1]}, IRISAN={len(common)}')
print('Irisan:', common)

only_ref = sorted(set(Xref.columns) - set(Xtgt.columns))
only_tgt = sorted(set(Xtgt.columns) - set(Xref.columns))
print('\nHanya CIC2017:', only_ref)
print('\nHanya CIC2018:', only_tgt)

Xref = Xref[common].copy()
Xtgt = Xtgt[common].copy()


In [ ]:
# === Simpan artefak + ringkasan skema untuk Tabel 1 ===
pickle.dump({'X': Xref, 'y': yref, 'features': common}, open(f'{OUT_DIR}/cic2017.pkl','wb'))
pickle.dump({'X': Xtgt, 'y': ytgt, 'features': common}, open(f'{OUT_DIR}/cic2018.pkl','wb'))

summary = {
    'seed': SEED,
    'max_rows_per_file': MAX_ROWS_PER_FILE,
    'ref_dataset': 'CIC-IDS2017',
    'tgt_dataset': 'CSE-CIC-IDS2018',
    'cic2017': {'rows': int(Xref.shape[0]), 'attack_pct': round(100*float(yref.mean()),2)},
    'cic2018': {'rows': int(Xtgt.shape[0]), 'attack_pct': round(100*float(ytgt.mean()),2)},
    'n_features_cic2017': int(cic17.shape[1]),
    'n_features_cic2018': int(cic18.shape[1]),
    'n_common_features': len(common),
    'common_features': common,
    'only_cic2017': only_ref,
    'only_cic2018': only_tgt,
}
json.dump(summary, open(f'{OUT_DIR}/schema_summary.json','w'), indent=2)
print('Tersimpan ke', OUT_DIR)
print(json.dumps({k:v for k,v in summary.items() if k!='common_features'}, indent=2))


In [ ]:
# === Upload artefak ke S3 (dijalankan di akhir tiap notebook) ===
subprocess.run(['aws', 's3', 'sync', OUT_DIR, S3_ARTIFACTS_OUT, '--only-show-errors'], check=True)
print('artifacts ter-upload ke', S3_ARTIFACTS_OUT)
